In [1]:
import numpy as np
import pandas as pd
import os

# Set seed for reproducibility
np.random.seed(42)

# Region code mapping to ensure unique prefixes for shark IDs
REGION_CODES = {
    "China": "CHN",
    "Pacific_Australia": "PAU",
    "Pacific_Baja": "PBJ",
    "Pacific_California": "PCA",
    "Pacific_Chile": "PCH",
    "Pacific_New_Zealand": "PNZ",
    "Southwest_SA": "SSA",
    "West_and_Central_SA": "WCSA",
    "Western_NA": "WNA",
    "Western_and_Central_NP": "WCNP",
}

In [2]:
PARAMETERS = [
    {"region": "China", "reference": "Hsu_2003", "sex": "M", "fl_min": 72.6, "fl_max": 250.9, "linf": 321.8, "k": 0.04, "t0": -6.07, "n_samples": 133, "longevity": 3},
    {"region": "China", "reference": "Hsu_2003", "sex": "F", "fl_min": 72.6, "fl_max": 314.9, "linf": 403.62, "k": 0.04, "t0": -5.27, "n_samples": 174, "longevity": None},
    {"region": "Pacific_Australia", "reference": "Chan_2001", "sex": "M", "fl_min": 66, "fl_max": 274, "linf": 267, "k": 0.31, "t0": -0.95, "n_samples": 24, "longevity": 9},
    {"region": "Pacific_Australia", "reference": "Chan_2001", "sex": "F", "fl_min": 74, "fl_max": 314, "linf": 349, "k": 0.15, "t0": -1.97, "n_samples": 52, "longevity": 17},
    {"region": "Pacific_Baja", "reference": "Ribot_Carballal_2005", "sex": "M", "fl_min": 68.6, "fl_max": 264, "linf": 375.4, "k": 0.05, "t0": -4.7, "n_samples": 109, "longevity": 55},
    {"region": "Pacific_Baja", "reference": "Ribot_Carballal_2005", "sex": "F", "fl_min": 68.6, "fl_max": 264, "linf": 375.4, "k": 0.05, "t0": -4.7, "n_samples": 109, "longevity": None},
    {"region": "Pacific_California", "reference": "Cailliet_Bedford_1983", "sex": "M", "fl_min": 80.6, "fl_max": 293, "linf": 298, "k": 0.07, "t0": -3.75, "n_samples": 44, "longevity": 38},
    {"region": "Pacific_California", "reference": "Cailliet_Bedford_1983", "sex": "F", "fl_min": 80.6, "fl_max": 293, "linf": 298, "k": 0.07, "t0": -3.75, "n_samples": 44, "longevity": None},
    {"region": "Pacific_Chile", "reference": "Cerna_Licandeo_2009", "sex": "M", "fl_min": 70, "fl_max": 258, "linf": 268.07, "k": 0.08, "t0": -3.58, "n_samples": 243, "longevity": None},
    {"region": "Pacific_Chile", "reference": "Cerna_Licandeo_2009", "sex": "F", "fl_min": 69, "fl_max": 300, "linf": 295.73, "k": 0.07, "t0": -3.18, "n_samples": 304, "longevity": None},
    {"region": "Pacific_New_Zealand", "reference": "Bishop_2006", "sex": "M", "fl_min": 100, "fl_max": 347, "linf": 302.2, "k": 0.05, "t0": -9.04, "n_samples": 145, "longevity": 48},
    {"region": "Southwest_SA", "reference": "Dono_2014", "sex": "M", "fl_min": 81, "fl_max": 250, "linf": 416, "k": 0.03, "t0": -6.18, "n_samples": 116, "longevity": None},
    {"region": "Southwest_SA", "reference": "Dono_2014", "sex": "F", "fl_min": 101, "fl_max": 330, "linf": 580, "k": 0.02, "t0": -7.52, "n_samples": 126, "longevity": None},
    {"region": "West_and_Central_SA", "reference": "This_study_fit1", "sex": "M", "fl_min": 79, "fl_max": 250, "linf": 328.74, "k": 0.08, "t0": -4.47, "n_samples": 129, "longevity": 23},
    {"region": "West_and_Central_SA", "reference": "This_study_fit1", "sex": "F", "fl_min": 73, "fl_max": 296, "linf": 407.65, "k": 0.04, "t0": -7.08, "n_samples": 109, "longevity": 28},
    {"region": "Western_NA", "reference": "Pratt_Casey_1983", "sex": "M", "fl_min": 69, "fl_max": 238, "linf": 302, "k": 0.26, "t0": -1, "n_samples": 49, "longevity": 10},
    {"region": "Western_NA", "reference": "Pratt_Casey_1983", "sex": "F", "fl_min": 69, "fl_max": 238, "linf": 345, "k": 0.2, "t0": -1, "n_samples": 54, "longevity": 14},
    {"region": "Western_NA", "reference": "Natanson_2006", "sex": "M", "fl_min": 72, "fl_max": 260, "linf": 253.3, "k": 0.12, "L0": 71.6, "n_samples": 118, "longevity": 21},
    {"region": "Western_NA", "reference": "Natanson_2006", "sex": "F", "fl_min": 64, "fl_max": 340, "linf": 365.6, "k": 0.08, "L0": 88.4, "n_samples": 140, "longevity": 38},
    {"region": "Western_and_Central_NP", "reference": "Semba_2009", "sex": "M", "fl_min": 73, "fl_max": 265, "linf": 255, "k": 0.16, "L0": 59.7, "n_samples": 128, "longevity": None},
    {"region": "Western_and_Central_NP", "reference": "Semba_2009", "sex": "F", "fl_min": 73, "fl_max": 330, "linf": 340, "k": 0.09, "L0": 59.7, "n_samples": 147, "longevity": None},
]

In [3]:
def vbgf_t0(t, linf, k, t0):
    return linf * (1 - np.exp(-k * (t - t0)))

def vbgf_L0(t, linf, k, L0):
    return linf - (linf - L0) * np.exp(-k * t)

def length_to_weight(length_cm):
    return 4.4e-6 * (length_cm ** 3.14)

def estimate_max_age(params):
    if params.get("longevity") is not None:
        return int(params["longevity"])
    k = params["k"]
    if "t0" in params:
        t0 = params["t0"]
        max_age = t0 - np.log(0.05) / k
    else:
        max_age = -np.log(0.05) / k
    return max(int(max_age), 25)

In [4]:
def generate_data_for_params(params, target_count):
    data = []
    linf, k, region, reference, sex = params["linf"], params["k"], params["region"], params["reference"], params["sex"]
    fl_min, fl_max = params["fl_min"], params["fl_max"]
    uses_L0 = "L0" in params
    max_age = estimate_max_age(params)

    # Beta distribution for ages (more young sharks)
    ages = np.random.beta(1.5, 2.5, target_count) * max_age
    
    for i, age in enumerate(ages):
        if uses_L0:
            length = vbgf_L0(age, linf, k, params["L0"])
        else:
            length = vbgf_t0(age, linf, k, params["t0"])
        
        length_clipped = np.clip(length, fl_min * 0.9, fl_max * 1.1)
        noise_factor = 1 + np.random.normal(0, 0.03)
        length_with_noise = np.clip(length_clipped * noise_factor, fl_min * 0.85, fl_max * 1.15)
        weight = length_to_weight(length_with_noise)
        age_rounded = round(age * 2) / 2
        
        data.append({
            "shark_id": "", # Placeholder
            "age_years": age_rounded,
            "length_FL_cm": round(length_with_noise, 1),
            "mass_kg": round(weight, 1),
            "sex": sex,
            "region": region,
            "source_study": reference,
            "L_inf_cm": linf,
            "K_per_year": k,
            "t0_or_L0": params.get("L0", params.get("t0")),
            "value_type": "L0" if uses_L0 else "t0"
        })
    return data

def generate_dataset(target_total=500):
    total_original_samples = sum(p["n_samples"] for p in PARAMETERS)
    all_data = []
    for params in PARAMETERS:
        weight = params["n_samples"] / total_original_samples
        target_count = max(10, int(target_total * weight))
        all_data.extend(generate_data_for_params(params, target_count))
    
    df = pd.DataFrame(all_data).sort_values(["region", "sex", "age_years"])
    
    # Assign new IDs
    for (region, sex), group in df.groupby(["region", "sex"]):
        region_code = REGION_CODES.get(region, region[:3].upper())
        for i, idx in enumerate(group.index):
            df.loc[idx, "shark_id"] = f"{region_code}_{sex}{i+1:03d}"
            
    return df.reset_index(drop=True)

In [5]:
# Genereer de data
df = generate_dataset(target_total=500)

# Toon samenvatting
print(f"Totaal aantal datapunten: {len(df)}")
print("\nAantal per regio:")
print(df.groupby("region").size())

# Toon de eerste paar rijen
df.head(10)

Totaal aantal datapunten: 498

Aantal per regio:
region
China                      60
Pacific_Australia          20
Pacific_Baja               42
Pacific_California         20
Pacific_Chile             108
Pacific_New_Zealand        29
Southwest_SA               48
West_and_Central_SA        46
Western_NA                 71
Western_and_Central_NP     54
dtype: int64


,shark_id,age_years,length_FL_cm,mass_kg,sex,region,source_study,L_inf_cm,K_per_year,t0_or_L0,value_type
0,CHN_F001,4.5,139.4,23.8,F,China,Hsu_2003,403.62,0.04,-5.27,t0
1,CHN_F002,5.0,132.4,20.2,F,China,Hsu_2003,403.62,0.04,-5.27,t0
2,CHN_F003,5.5,142.0,25.2,F,China,Hsu_2003,403.62,0.04,-5.27,t0
3,CHN_F004,10.0,185.8,58.7,F,China,Hsu_2003,403.62,0.04,-5.27,t0
4,CHN_F005,12.0,202.7,77.1,F,China,Hsu_2003,403.62,0.04,-5.27,t0
5,CHN_F006,13.0,214.9,92.6,F,China,Hsu_2003,403.62,0.04,-5.27,t0
6,CHN_F007,13.5,220.5,100.4,F,China,Hsu_2003,403.62,0.04,-5.27,t0
7,CHN_F008,14.0,215.4,93.2,F,China,Hsu_2003,403.62,0.04,-5.27,t0
8,CHN_F009,15.0,230.3,115.0,F,China,Hsu_2003,403.62,0.04,-5.27,t0
9,CHN_F010,18.0,249.4,147.8,F,China,Hsu_2003,403.62,0.04,-5.27,t0


In [ ]:
output_file = "mako_growth_data_by_region.csv"
df.to_csv(output_file, index=False)
print(f"Bestand opgeslagen als: {output_file}")